In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
from datetime import datetime, timedelta
import requests
import io

In [2]:
#Importar planes de compra en formato csv
df_plan_panaderia = pd.read_csv(r"C:\Users\esteban.correa\Downloads\PLAN_PANADERIA.csv")
df_plan_congelados = pd.read_csv(r"C:\Users\esteban.correa\Downloads\PLAN_CONGELADOS.csv")
df_plan_compra_general = pd.read_csv(r"C:\Users\esteban.correa\Downloads\PLAN_COMPRA_GENERAL.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\juane\\Downloads\\PLAN_PANADERIA.csv'

In [ ]:
#Conectar a snowflake con las credenciales
snowflake_connection = snowflake.connector.connect(user='esteban.correa@RAPPI.COM', 
                                   authenticator='externalbrowser', 
                                   account='hg51401', 
                                   warehouse="RP_PERSONALUSER_WH",
                                   database="FIVETRAN")

In [ ]:
#Definir variable para la query de incoming stock
query_incoming_stock = """
SELECT 
    CONCAT(WAREHOUSE_ID, '-', PRODUCT_ID) AS WAREHOUSE_PRODUCT_ID,
    SUM(QUANTITY - RECEIVED) AS INCOMING_STOCK_SF
FROM 
    fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER a
LEFT JOIN 
    fivetran.CO_AMYSQL_TURBO_EMERGENCY_ORDER_TURBO_PO_SAVVY_MS.PURCHASE_ORDER_DETAIL b 
    ON b.PURCHASE_ORDER_ID = a.id
WHERE 
    a.STATUS_ID = 5
GROUP BY 
    WAREHOUSE_PRODUCT_ID
"""

In [ ]:
#Ejecutar query de incoming stock en snowflake
df_incoming_stock = pd.read_sql(query_incoming_stock, snowflake_connection)

In [ ]:
#Cocatenar en uno los diferentes planes de compra
df_plan_externos = pd.concat([df_plan_panaderia, df_plan_congelados, df_plan_compra_general])
df_plan_externos = df_plan_externos.reset_index(drop=True)

In [ ]:
#Crear columna warehouse product id
df_plan_externos['WAREHOUSE_PRODUCT_ID'] = df_plan_externos['Warehouseid'].astype(str) + '-' + df_plan_fruver_colombia['Sku Id'].astype(str)

In [ ]:
#Realizar un left join con el data frame que contiene el incoming stock proveniente de snowflake
df_plan_externos = pd.merge(df_plan_externos, df_incoming_stock, on='WAREHOUSE_PRODUCT_ID', how='left')

In [ ]:
#Crear columna Incoming stock difference y asignarle la diferencia entre el incoming stock proveniente de F9 y el proveniente de snowflake
df_plan_externos['INCOMING STOCK DIFFERENCE'] = df_plan_externos['Incoming Stock'] - df_plan_externos['INCOMING_STOCK_SF']

In [ ]:
#Crear un dataframe que contenga un mapeo de warehouse id, city id y city name
warehouse_city_mapping = pd.DataFrame({
    'Warehouseid': [49, 63, 66, 84, 107, 21, 25, 27, 29, 31, 48, 52, 54, 55, 68, 80, 87, 91, 92, 3, 4, 5, 6, 7, 108, 109, 2, 75, 79, 90, 93, 8, 9, 11, 12, 14, 16, 17, 18, 19, 20, 23, 24, 26, 28, 30, 33, 47, 50, 51, 53, 56, 58, 60, 61, 67, 69, 70, 72, 73, 78, 110, 103, 104, 106, 105],
    'CITY_ID': [3, 3, 3, 5, 5, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 35, 35, 35, 35, 35, 35, 35, 36, 36, 36, 36, 36, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 37, 40, 42, 42, 43],
    'CITY_NAME': ['Barranquilla', 'Barranquilla', 'Barranquilla', 'Bucaramanga', 'Bucaramanga', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Medellín', 'Cartagena', 'Barranquilla', 'Cartagena', 'Cartagena', 'Cartagena', 'Santa Marta', 'Villavicencio', 'Cali', 'Cali', 'Cali', 'Cali', 'Cali', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Bogotá', 'Armenia', 'Pereira', 'Ibagué', 'Montería']
})

In [ ]:
#Realizar un left join con el dataframe que contiene un mapeo de warehouse id, city id y city name
df_plan_externos = df_plan_externos.merge(warehouse_city_mapping, on='Warehouseid', how='left')


In [ ]:
multiples_restrictions = {
    '77': 5, #Pan pa ya
    '316': 5, #STLTH
    '204':5, #La Boutique
    '45':5, #New Pork
    '210':10, #Pod Salt
    '295':1, #Luisa Postres
    '168':10, #Glucloud
    '4':5, #Inversiones Liev
    '200':5, #Arte Dolce
    '135': 6, #Sr Wok
    '305':12, #Rosadela
    '318':10, #Evapify
    '185':5, #Relx
    '144':12, #Productos Donald's
    '313':{ #Notco
        '14467':32,
        '14469':32,
        '14472':18,
        '14473':56
        },
    '119': { #Pixie
        '3555': 20,
        '3320': 20,
        '3468': 20,
        '3587': 20
    }
}

In [ ]:
#Redondea las cantidades al multiplo mas cercano (hacia arriba)
def round_up_to_multiple(quantity, multiple):
    if quantity % multiple == 0:
        return quantity
    else:
        return np.ceil(quantity / multiple) * multiple

In [ ]:
#Crear Columna adjusted quantity y asignarle el resultado de redondear los valores de compra al multiplo especificado
def apply_multiples_restrictions(df, multiples_restrictions):
    df['ADJUSTED QUANTITY'] = df['Purchase Quantity']
    
    for supplier, restriction in multiples_restrictions.items():
        supplier_df = df[df['Primary Supplier Id'] == supplier]
        
        if isinstance(restriction, dict):
            #Proveedores con diferentes multiplos por producto
            for product, multiple in restriction.items():
                product_df = supplier_df[supplier_df['Sku Id'] == product]
                for idx, row in product_df.iterrows():
                    adjusted_quantity = round_up_to_multiple(row['Purchase Quantity'], multiple)
                    df.loc[idx, 'ADJUSTED QUANTITY'] = adjusted_quantity
        else:
            #El mismo multiplo para todos los productos
            for idx, row in supplier_df.iterrows():
                adjusted_quantity = round_up_to_multiple(row['Purchase Quantity'], restriction)
                df.loc[idx, 'ADJUSTED QUANTITY'] = adjusted_quantity
                
    return df

In [ ]:
supplier_restrictions = {
    '316': [ #STLTH
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    '57': [ #Vive Agro
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    '178': [ #Kampos carnicos
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 10, 'max_quantity': 5000},
    ],
    '313': [ #Notco
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
    '166': [ #Selvati
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 20, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 20, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 20, 'max_quantity': 5000}
    ],
   '210': [ #Pod Salt
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 56, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000}, ## revisar scheduler ya que esta con otro id
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
   ],
    '168':[ #Glucloud
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 50, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 50, 'max_quantity': 5000}
    ],
    '4':[ #Inversiones Liev
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 10, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 10, 'max_quantity': 5000}
    ],
    '200':[ #Arte Dolce
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 40, 'max_quantity': 5000},
         {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 40, 'max_quantity': 5000}
    ],
    '135': [ #Sr Wok
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 18, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 18, 'max_quantity': 5000}
    ],
    '185':[ #RELX
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 25, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 25, 'max_quantity': 5000}
    ],
    '318': [ #Evapify
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 4, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 16, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 31, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 49, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 58, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 60, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 61, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 63, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 66, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 78, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 80, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 84, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 103, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 104, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 105, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 106, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 107, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 108, 'min_quantity': 30, 'max_quantity': 5000},
        {'type': 'quantity', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_quantity': 30, 'max_quantity': 5000}
    ],
    '77':[ #Pan pa ya
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 2, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 3, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 5, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 6, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 7, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 8, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 9, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 11, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 12, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 14, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 17, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 18, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 19, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 21, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 23, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 24, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 25, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 26, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 27, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 28, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 29, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 33, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 47, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 48, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 50, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 51, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 52, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 55, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 67, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 68, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 70, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 72, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 73, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 79, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 90, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 91, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 93, 'min_value': 70000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'warehouse', 'WAREHOUSE_ID': 110, 'min_value': 70000, 'max_value': 10000000}
    ],
    '119' :[ #Pixie
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 35, 'min_value': 800000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 3, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 5, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 7, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 36, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 37, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 40, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 42, 'min_value': 350000, 'max_value': 10000000},
        {'type': 'monetary', 'level': 'city', 'CITY_ID': 43, 'min_value': 350000, 'max_value': 10000000}
    ],
    
}
    
        
    

In [ ]:
#Levels: product,warehouse,city,total
#types: quantity, monetary

def check_supplier_restrictions(df, supplier_restrictions, multiples_restrictions):
    df['violation'] = False
    df['violation_description'] = ''
    
    # Apply multiples restrictions and adjust quantities
    df = apply_multiples_restrictions(df, multiples_restrictions)
    
    for supplier, restrictions in supplier_restrictions.items():
        supplier_df = df[df['PRIMARY_SUPPLIER_ID'] == supplier]
        print(f"Checking restrictions for supplier: {supplier}")  # Debug
        print(f"Supplier DataFrame:\n{supplier_df}")  # Debug
        
        if supplier_df.empty:
            print(f"  No data for supplier: {supplier}")  # Debug
            continue
        
        for restriction in restrictions:
            restriction_type = restriction['type']
            level = restriction['level']
            
            if level == 'warehouse':
                group_col = 'Warehouseid'
            elif level == 'city':
                group_col = 'City ID'
            elif level == 'product':
                group_col = 'Sku Id'
            else: 
                group_col = None
                
            if group_col:
                if group_col not in supplier_df.columns:
                    print(f"  Column {group_col} not in supplier DataFrame")  # Debug
                    continue

                grouped = supplier_df.groupby(group_col)
                print(f"Grouping by {group_col}: {grouped.groups.keys()}")  # Debug
                
                if not grouped.groups:
                    print(f"  No groups found for {group_col} in supplier DataFrame")  # Debug
                    continue
                
                for key, group in grouped:
                    print(f"    Grouping by {group_col}, Key: {key}")  # Debug
                    if restriction_type == 'quantity':
                        total_quantity = group['adjusted_quantity'].sum()
                        print(f"      Total quantity: {total_quantity}")  # Debug
                        if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                            df.loc[group.index, 'violation'] = True
                            df.loc[group.index, 'violation_description'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
                    elif restriction_type == 'monetary':
                        total_value = (group['adjusted_quantity'] * group['PRIMARY_SUPPLIER_PRICE']).sum()
                        print(f"      Total value: {total_value}")  # Debug
                        if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                            df.loc[group.index, 'violation'] = True
                            df.loc[group.index, 'violation_description'] = f'{restriction_type} violation at {level} level for {group_col}={key}'
            else: 
                if restriction_type == 'quantity':
                    total_quantity = supplier_df['adjusted_quantity'].sum()
                    print(f"    Total quantity: {total_quantity}")  # Debug
                    if total_quantity < restriction['min_quantity'] or total_quantity > restriction['max_quantity']:
                        df.loc[supplier_df.index, 'violation'] = True
                        df.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
                elif restriction_type == 'monetary':
                    total_value = (supplier_df['adjusted_quantity'] * supplier_df['PRIMARY_SUPPLIER_PRICE']).sum()
                    print(f"    Total value: {total_value}")  # Debug
                    if total_value < restriction['min_value'] or total_value > restriction['max_value']:
                        df.loc[supplier_df.index, 'violation'] = True
                        df.loc[supplier_df.index, 'violation_description'] = f'{restriction_type} violation at total level'
    
    return df

In [ ]:
df_purchase_plans_with_violations = check_supplier_restrictions(df_purchase_plan, supplier_restrictions, multiples_restrictions)

In [ ]:
df_purchase_plans_with_violations.to_csv(r"C:\Users\juane\Downloads\PURCHASE_PLAN_COLOMBIA.csv")